# Data Quality Checks

## Metadata check

### Setup

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

RAW = Path().resolve().parent / 'data' / 'raw'
files = sorted(RAW.glob('*/*.csv'))

print(f'{len(files)} files under {RAW}')

64 files under C:\Users\simon\OneDrive\Tech\Github\ml-shared-bike\data\raw


### File inventory

In [3]:
# One row per file: year, name, size, row count, encoding marker
def sniff_encoding(path):
    head = path.open('rb').read(4)
    if head.startswith(bytes([0xEF, 0xBB, 0xBF])):  # UTF-8 BOM
        return 'utf-8-sig'
    return 'utf-8'


def count_rows(path, encoding):
    with path.open('r', encoding=encoding, errors='replace') as fh:
        return sum(1 for _ in fh) - 1  # minus header


inventory = pd.DataFrame(
    [
        {
            'year': path.parent.name,
            'file': path.name,
            'mb': round(path.stat().st_size / 1024**2, 2),
            'encoding': (enc := sniff_encoding(path)),
            'rows': count_rows(path, enc),
        }
        for path in files
    ]
)

inventory

,year,file,mb,encoding,rows
0,2019,2019-Q1.csv,23.90,utf-8-sig,189063
1,2019,2019-Q2.csv,82.76,utf-8-sig,651685
2,2019,2019-Q3.csv,143.36,utf-8-sig,1130353
3,2019,2019-Q4.csv,59.08,utf-8-sig,468416
4,2020,2020-01.csv,12.86,utf-8-sig,102148
...,...,...,...,...,...
59,2025,bikeshare_2025_08.csv,152.30,utf-8,1164697
60,2025,bikeshare_2025_09.csv,138.81,utf-8,1062001
61,2025,bikeshare_2025_10.csv,113.73,utf-8,872482
62,2025,bikeshare_2025_11.csv,66.68,utf-8,511934


### Files and rows per year

In [4]:
# Files and rows per year - expect 12 monthly files, 2019 is quarterly
inventory.groupby('year').agg(
    files=('file', 'size'),
    rows=('rows', 'sum'),
    mb=('mb', 'sum'),
)

,files,rows,mb
year,,,
2019,4,2439517,309.10
2020,12,2911308,371.32
2021,12,3575182,455.18
2022,11,4300240,537.60
2023,12,5713141,696.07
2024,1,6953094,924.18
2025,12,7812520,1021.37


### Schema drift across files

In [5]:
# Header of every file, to expose schema drift across years
def read_header(path):
    enc = sniff_encoding(path)
    return tuple(pd.read_csv(path, nrows=0, encoding=enc).columns)


headers = pd.DataFrame(
    [{'year': p.parent.name, 'file': p.name, 'header': read_header(p)} for p in files]
)

# Group files by the exact column tuple they use
schemas = (
    headers.groupby('header')
    .agg(files=('file', 'size'), years=('year', lambda s: sorted(set(s))))
    .reset_index()
)

for i, row in schemas.iterrows():
    print(f'--- schema {i}: {row["files"]} files, years {row["years"]}')
    print('   ', list(row['header']))

--- schema 0: 51 files, years ['2019', '2020', '2021', '2022', '2023']
    ['Trip Id', 'Trip  Duration', 'Start Station Id', 'Start Time', 'Start Station Name', 'End Station Id', 'End Time', 'End Station Name', 'Bike Id', 'User Type']
--- schema 1: 13 files, years ['2024', '2025']
    ['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time', 'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name', 'Bike_Id', 'User_Type', 'Bike_Model']


### Column presence by schema

In [6]:
# Which columns appear in which schema, and the dtypes pandas infers
all_cols = sorted({c for h in schemas['header'] for c in h})
presence = pd.DataFrame(
    {f'schema {i}': [c in h for c in all_cols] for i, h in enumerate(schemas['header'])},
    index=all_cols,
)

presence

,schema 0,schema 1
Bike Id,True,False
Bike_Id,False,True
Bike_Model,False,True
End Station Id,True,False
End Station Name,True,False
End Time,True,False
End_Station_Id,False,True
End_Station_Name,False,True
End_Time,False,True
Start Station Id,True,False


### Inferred dtypes per schema

In [7]:
# Inferred dtypes from a sample of each schema, plus a preview
for i, header in enumerate(schemas['header']):
    sample_name = headers.loc[headers['header'] == header, 'file'].iloc[0]
    sample = next(p for p in files if p.name == sample_name)
    df = pd.read_csv(sample, nrows=5000, encoding=sniff_encoding(sample))
    print(f'--- schema {i}: {sample.parent.name}/{sample.name}')
    print(df.dtypes.to_string())
    print()

--- schema 0: 2019/2019-Q1.csv
Trip Id               int64
Trip  Duration        int64
Start Station Id      int64
Start Time              str
Start Station Name      str
End Station Id        int64
End Time                str
End Station Name        str
Bike Id               int64
User Type               str

--- schema 1: 2024/bikeshare-ridership-2024.csv
Trip_Id                 int64
Trip_Duration           int64
Start_Station_Id        int64
Start_Time                str
Start_Station_Name        str
End_Station_Id        float64
End_Time                  str
End_Station_Name          str
Bike_Id                 int64
User_Type                 str
Bike_Model                str



### Duplicate files

In [8]:
# Byte-level duplicate files - the same data delivered twice
inventory['sha256'] = [
    hashlib.sha256(p.read_bytes()).hexdigest() for p in files
]
inventory[inventory.duplicated('sha256', keep=False)].sort_values('sha256')

,year,file,mb,encoding,rows,sha256


## Validity check

## Completeness check

## Uniqueness check

## Consistency check

## Accuracy check

## Timeliness check

## Summary